In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from scipy.stats import wilcoxon

# ==========================================
adata = CRC_subcluster_annot
ref_types   = ["Proliferating tumor", "Invasive tumor"]  
target_type = "Fibroblasts"                                    

radii = np.arange(10, 51, 5)  # 10, 15, ..., 50

batches = sorted(
    adata.obs["batch"].astype(str).unique(),
    key=lambda x: int(x) if str(x).isdigit() else x
)

def nms_curve_for_sample(coords, labels, ref_label, target_label, radii):
    """
    coords: (n,2) float array
    labels: (n,) array-like of str (cell_type_final)
    radii : list/array of radii
    return: (NaN-safe)
    """
    labels = np.asarray(labels)

    mask_ref = (labels == ref_label)
    mask_tgt = (labels == target_label)

    ref_pts = coords[mask_ref]
    tgt_pts = coords[mask_tgt]

    n_ref, n_tgt = len(ref_pts), len(tgt_pts)
    if (n_ref <= 1) or (n_tgt == 0):
        return np.full(len(radii), np.nan, dtype=float)

    tree_ref = cKDTree(ref_pts)
    tree_tgt = cKDTree(tgt_pts)

    out = []
    for r in radii:
        neigh_rt = tree_ref.query_ball_tree(tree_tgt, r)
        n_rt = sum(len(L) for L in neigh_rt)

        neigh_rr = tree_ref.query_ball_tree(tree_ref, r)
        n_rr = sum(max(0, len(L) - 1) for L in neigh_rr)

        if n_rr == 0:
            out.append(np.nan)
        else:
            nms = n_rt * (n_ref - 1) / (2.0 * n_rr * n_tgt)
            out.append(nms)

    return np.asarray(out, dtype=float)

# ==========================================
coords_all = np.asarray(adata.obsm["spatial"], dtype=np.float32)[:, :2]
labels_all = adata.obs["cell_type_tmp"].astype(str).values
batch_all  = adata.obs["batch"].astype(str).values

curves_ref1 = []  # Proliferating tumor vs CAF
curves_ref2 = []  # Invasive tumor vs CAF

for b in batches:
    idx = (batch_all == str(b))
    coords_b = coords_all[idx]
    labels_b = labels_all[idx]

    c1 = nms_curve_for_sample(coords_b, labels_b, ref_types[0], target_type, radii)
    c2 = nms_curve_for_sample(coords_b, labels_b, ref_types[1], target_type, radii)

    curves_ref1.append(c1)
    curves_ref2.append(c2)

curves_ref1 = np.vstack(curves_ref1)  # (n_batches, n_radii)
curves_ref2 = np.vstack(curves_ref2)

# ==========================================
def mean_sem(arr):
    mean = np.nanmean(arr, axis=0)
    n = np.sum(np.isfinite(arr), axis=0)
    sem = np.nanstd(arr, axis=0, ddof=1) / np.sqrt(np.maximum(n, 1))
    return mean, sem

mean1, sem1 = mean_sem(curves_ref1)
mean2, sem2 = mean_sem(curves_ref2)

# ==========================================
pvals = []
n_used = []

for j in range(len(radii)):
    a = curves_ref1[:, j]
    b = curves_ref2[:, j]
    m = np.isfinite(a) & np.isfinite(b)
    n_used.append(int(m.sum()))

    if m.sum() < 3:
        pvals.append(np.nan)
        continue

    if np.allclose(a[m], b[m]):
        pvals.append(1.0)
        continue

    try:
        p = wilcoxon(a[m], b[m], zero_method="wilcox", alternative="two-sided").pvalue
    except ValueError:
        p = np.nan
    pvals.append(p)

def p_to_star(p):
    if not np.isfinite(p): return ""
    if p < 1e-3: return "***"
    if p < 1e-2: return "**"
    if p < 5e-2: return "*"
    return ""

stars = [p_to_star(p) for p in pvals]

# ==========================================
plt.style.use("default")

fig, ax = plt.subplots(figsize=(6, 3.5), dpi=200)
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

c1 = "#FF0000"   # Proliferating tumor
c2 = "#FFA500"   # Invasive tumor

ax.errorbar(
    radii, mean1, yerr=sem1,
    marker="o", capsize=3, lw=2.0, color=c1,
    label="Proliferating tumor"
)
ax.errorbar(
    radii, mean2, yerr=sem2,
    marker="o", capsize=3, lw=2.0, color=c2,
    label="Invasive tumor"
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.spines["left"].set_linewidth(1.6)
ax.spines["bottom"].set_linewidth(1.6)
ax.spines["left"].set_color("black")
ax.spines["bottom"].set_color("black")

ax.tick_params(
    axis="both", which="major",
    direction="out", length=6, width=1.4,
    colors="black", labelsize=14
)

ax.grid(False)

step = radii[1] - radii[0]
ax.set_xlim(radii.min() - step * 0.6, radii.max() + step * 0.6)
ax.set_xticks(radii)
ax.set_xticklabels([str(r) for r in radii], color="black")

ax.set_xlabel("Radius", fontsize=16, color="black")
ax.set_ylabel("Co-localization scores", fontsize=16, color="black")

y_top = np.nanmax(np.vstack([mean1 + sem1, mean2 + sem2]), axis=0)
y_bottom = np.nanmin(np.vstack([mean1 - sem1, mean2 - sem2]), axis=0)
y_range = float(np.nanmax(y_top) - np.nanmin(y_bottom))
offset = 0.03 * y_range if np.isfinite(y_range) and y_range > 0 else 0.005

for x, y, s in zip(radii, y_top, stars):
    if s:
        ax.text(x, y + offset, s, ha="center", va="bottom", fontsize=12, color="black")

ax.set_title("Co-localization Scores with Fibroblasts", fontsize=16, fontweight="bold", pad=10)
ax.legend(frameon=False, loc="upper left", fontsize=14)

fig.tight_layout()
plt.show()


import pandas as pd
import numpy as np

df_p = pd.DataFrame({
    "radius": radii,
    "n_pairs_used": n_used,      # 각 반경에서 wilcoxon에 실제 사용된 batch 쌍 개수
    "p_value": pvals,
    "star": stars,
    "mean(Prolif)": mean1,
    "mean(Invasive)": mean2,
    "diff(Invasive-Prolif)": mean2 - mean1,
})

pd.set_option("display.max_rows", None)